# Cincinnati Trails: CAGIS vs. OpenStreetMap

This notebook loads the trail sources, maps their coverage, exports both datasets as geometrically matched runs with counterpart attributes, and separately interprets their tags for human review. Discrepancies are candidates, not verified errors.

Step 5 produces neutral GeoJSON layers; step 6 owns interpretation and presentation. See [symbology_rules.md](symbology_rules.md) for portable rule definitions. Source downloads are cached as `<resource>_YYYY-MM-DD.geojson` using the local fetch date. Each source reuses only today's cache; if it is missing or `REFRESH` is enabled, the source is downloaded. Older daily caches are retained. Historical OSM snapshot timestamps remain in the resource name, separately from the fetch date.

## 1. Set options and load CAGIS trails

In [ ]:
%pip install geopandas plotly shapely requests matplotlib

from datetime import datetime, timezone
import json
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import requests
import shapely
from shapely.ops import substring


def geojson_payload_without_nulls(gdf):
    import numpy as np

    def encode_value(value):
        if isinstance(value, (datetime, pd.Timestamp, np.datetime64)):
            return None if pd.isna(value) else pd.Timestamp(value).isoformat()
        if isinstance(value, np.generic):
            return value.item()
        raise TypeError(f"Unsupported GeoJSON property type: {type(value).__name__}")

    payload = json.loads(gdf.to_json(na="drop", drop_id=True, default=encode_value))
    for feature in payload["features"]:
        feature["properties"] = {
            key: value for key, value in feature.get("properties", {}).items()
            if value is not None
        }
    return payload


def write_geojson_without_nulls(gdf, path):
    payload = geojson_payload_without_nulls(gdf)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, allow_nan=False), encoding="utf-8")


def daily_cache_path(resource):
    return CACHE_DIR / f"{resource}_{datetime.now():%Y-%m-%d}.geojson"


CAGIS_URL = (
    "https://cagisonline.hamilton-co.org/arcgis/rest/services/"
    "Cincinnati_Park_Board/CPB_Natural_Resource/FeatureServer/2"
)
PARKS_URL = (
    "https://cagisonline.hamilton-co.org/arcgis/rest/services/"
    "Cincinnati_Park_Board/CPB_Planning/FeatureServer/15"
)
OVERPASS_URLS = (
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
)
HEADERS = {"User-Agent": "trail-discrepancy-report/0.1 (personal civic project)"}
OSM_HIGHWAYS = "path|footway|service|track|steps|bridleway|cycleway"

PROJECT_DIR = Path.cwd()
CACHE_DIR = PROJECT_DIR / "cache"
REFRESH = False  # Set True to re-download data from the source APIs.
# OVERPASS_DATE = "2025-04-22T00:00:00Z"  # Optional ISO date/time, e.g. "2025-04-22T00:00:00Z".
OVERPASS_DATE = None

# Comparison settings (all distances are metres).
TOL_M = 10
STEP_M = 20
MIN_COVER = 0.80
PARKS_BUFFER_M = 0

In [ ]:
cagis_path = daily_cache_path("cagis_trails")
if cagis_path.exists() and not REFRESH:
    print(f"  using cached {cagis_path}")
    cagis = gpd.read_file(cagis_path)
else:
    features, offset, previous_first = [], 0, None
    for _ in range(200):
        response = requests.get(
            f"{CAGIS_URL}/query",
            params={
                "where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson",
                "resultOffset": offset, "resultRecordCount": 1000,
            },
            headers=HEADERS, timeout=90, verify=False,
        )
        response.raise_for_status()
        batch = response.json().get("features", [])
        if not batch:
            break
        first = json.dumps(batch[0], sort_keys=True)
        if first == previous_first:
            break
        previous_first = first
        features.extend(batch)
        offset += len(batch)
    if not features:
        raise RuntimeError("CAGIS query returned no features (check URL / service status).")
    cagis = gpd.GeoDataFrame.from_features(features, crs=4326)
    cagis_path.parent.mkdir(parents=True, exist_ok=True)
    cagis.to_file(cagis_path, driver="GeoJSON")

fields = [column for column in cagis.columns if column != "geometry"]
id_field = next((column for needle in ("objectid", "trailid", "id") for column in fields if needle in column.lower()), None)
name_field = next((column for needle in ("trailname", "name") for column in fields if needle in column.lower()), None)
edit_field = next((column for needle in ("last_edit", "edited", "editdate", "modified") for column in fields if needle in column.lower()), None)

print(f"CAGIS: {len(cagis):,} features; CRS={cagis.crs}")
print(f"Detected fields: id={id_field}, name={name_field}, last edit={edit_field}")

## 2. Load park boundaries and OSM trails

Load the Cincinnati Park Board boundary layer, query OSM over the parks' combined extent, and retain complete ways that intersect CPB-maintained park boundaries buffered outward by 2 metres.

In [ ]:
parks_path = daily_cache_path("cincinnati_parks")
if parks_path.exists() and not REFRESH:
    print(f"  using cached {parks_path}")
    parks = gpd.read_file(parks_path)
else:
    features, offset, previous_first = [], 0, None
    for _ in range(200):
        response = requests.get(
            f"{PARKS_URL}/query",
            params={
                "where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson",
                "resultOffset": offset, "resultRecordCount": 1000,
            },
            headers=HEADERS, timeout=90, verify=False,
        )
        response.raise_for_status()
        batch = response.json().get("features", [])
        if not batch:
            break
        first = json.dumps(batch[0], sort_keys=True)
        if first == previous_first:
            break
        previous_first = first
        features.extend(batch)
        offset += len(batch)
    if not features:
        raise RuntimeError("Park-boundary query returned no features (check URL / service status).")
    parks = gpd.GeoDataFrame.from_features(features, crs=4326)
    parks_path.parent.mkdir(parents=True, exist_ok=True)
    parks.to_file(parks_path, driver="GeoJSON")

parks = parks.to_crs("EPSG:4326")
print(f"Park boundary features: {len(parks):,}")
print(f"Park extent (minx, miny, maxx, maxy): {parks.total_bounds}")

### Fetch OSM ways and pedestrian areas within buffered park boundaries

The Overpass request uses the combined park extent. Retain parks maintained by CPB or intersecting at least one CAGIS trail. Unspecified maintenance alone does not qualify a park. OSM ways and pedestrian areas are retained when they intersect these selected park boundaries buffered outward by `PARKS_BUFFER_M` metres in a projected metric CRS. The geometries are not clipped; pedestrian areas are later used directly as matching buffers.

In [ ]:
snapshot = None
if OVERPASS_DATE is not None:
    snapshot_time = datetime.fromisoformat(str(OVERPASS_DATE).replace("Z", "+00:00"))
    if snapshot_time.tzinfo is None:
        snapshot_time = snapshot_time.replace(tzinfo=timezone.utc)
    snapshot = snapshot_time.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

park_bbox = tuple(parks.total_bounds + [-0.002, -0.002, 0.002, 0.002])
osm_cache_name = "osm_trails_parks_extent"
if snapshot:
    osm_cache_name += f"_{snapshot.replace('-', '').replace(':', '').replace('T', '_')}"
osm_raw_path = daily_cache_path(osm_cache_name)

if osm_raw_path.exists() and not REFRESH:
    print(f"  using cached {osm_raw_path}")
    osm_raw = gpd.read_file(osm_raw_path)
else:
    west, south, east, north = park_bbox
    date_clause = f'[date:"{snapshot}"]' if snapshot else ""
    query = (
        f'[out:json][timeout:360]{date_clause}[bbox:{south},{west},{north},{east}];'
        f'('
        f'  way["highway"~"^({OSM_HIGHWAYS}|proposed|abandoned|disused)$"];'
        f'  way["disused:highway"];'
        f'  way["abandoned:highway"];'
        f'  way["proposed:highway"];'
        f'  way["highway"="pedestrian"]["area"="yes"];'
        f'  relation["type"="multipolygon"]["highway"="pedestrian"];'
        f');'
        f'out geom tags meta;'
    )
    last_error = None
    for endpoint in OVERPASS_URLS:
        try:
            print(f"Querying Overpass endpoint: {endpoint}")
            response = requests.post(
                endpoint, data={"data": query}, headers=HEADERS, timeout=360, verify=False
            )
            response.raise_for_status()
            payload = response.json()
            if payload.get("remark"):
                raise ValueError(f"Incomplete Overpass response: {payload['remark']}")
            break
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            print(f"  endpoint failed: {exc}")
    else:
        raise RuntimeError("All Overpass endpoints failed.") from last_error

    def pedestrian_area_geometry(element):
        from shapely.ops import polygonize

        if element["type"] == "way":
            points = [(point["lon"], point["lat"]) for point in element.get("geometry", [])]
            geometry = shapely.Polygon(points)
        else:
            rings = {"outer": [], "inner": []}
            for member in element.get("members", []):
                role = member.get("role") or "outer"
                if member.get("type") == "way" and role in rings:
                    points = [(point["lon"], point["lat"]) for point in member.get("geometry", [])]
                    if len(points) < 2:
                        raise ValueError(f"Incomplete pedestrian area relation {element['id']}")
                    rings[role].append(shapely.LineString(points))
            outer = list(polygonize(rings["outer"]))
            inner = list(polygonize(rings["inner"]))
            if not outer:
                raise ValueError(f"No closed outer ring for pedestrian area relation {element['id']}")
            geometry = shapely.union_all(outer).difference(shapely.union_all(inner))
        geometry = shapely.make_valid(geometry)
        polygons = [part for part in shapely.get_parts(geometry) if part.geom_type in {"Polygon", "MultiPolygon"}]
        if not polygons:
            raise ValueError(f"No polygon geometry for pedestrian area {element['id']}")
        return shapely.union_all(polygons)

    rows, seen_ids = [], set()
    for element in payload.get("elements", []):
        osm_id = element["id"]
        if (element["type"], osm_id) in seen_ids:
            continue
        seen_ids.add((element["type"], osm_id))
        tags = element.get("tags", {})
        requires_vis = (
            tags.get("footway") in ("sidewalk", "crossing")
            or tags.get("highway") == "service"
        )
        if requires_vis and not tags.get("trail_visibility", "").strip():
            continue
        is_pedestrian_area = tags.get("highway") == "pedestrian" and (
            element["type"] == "relation" or tags.get("area") == "yes"
        )
        if is_pedestrian_area:
            geometry = pedestrian_area_geometry(element)
        else:
            points = [(point["lon"], point["lat"]) for point in element.get("geometry", [])]
            if len(points) < 2:
                continue
            geometry = shapely.LineString(points)
        rows.append({
            "osm_type": element["type"],
            "osm_id": osm_id,
            "osm_name": tags.get("name"),
            "osm_highway": tags.get("highway"),
            "osm_last_edit": element.get("timestamp"),
            **{
                f"osm_{key}": value
                for key, value in tags.items()
                if key not in {"name", "highway"}
            },
            "geometry": geometry,
        })

    columns = ["osm_type", "osm_id", "osm_name", "osm_highway", "osm_last_edit", "osm_check_date", "geometry"]
    osm_raw = gpd.GeoDataFrame(rows, columns=None if rows else columns, geometry="geometry", crs=4326)
    write_geojson_without_nulls(osm_raw, osm_raw_path)

def select_comparison_parks(parks, cagis):
    maintained = parks["MAINTBY"].str.contains("CPB", case=False, na=False).to_numpy(dtype=bool)
    if not parks.empty and not cagis.empty:
        _, park_indices = parks.sindex.query(cagis.to_crs(parks.crs).geometry, predicate="intersects")
        maintained[park_indices] = True
    return parks.loc[maintained].copy()


comparison_parks = select_comparison_parks(parks, cagis)
if osm_raw.empty or comparison_parks.empty:
    osm = osm_raw.iloc[0:0].copy()
else:
    metric_crs = comparison_parks.estimate_utm_crs()
    park_area = shapely.union_all(comparison_parks.to_crs(metric_crs).geometry.values).buffer(PARKS_BUFFER_M)
    osm_metric = osm_raw.to_crs(metric_crs)
    osm = osm_raw.loc[osm_metric.geometry.intersects(park_area)].reset_index(drop=True)

print(f"OSM features fetched in park extent: {len(osm_raw):,}")
print(f"Complete OSM features intersecting CPB-maintained parks or parks with CAGIS trails + {PARKS_BUFFER_M} m buffer: {len(osm):,}")

## 3. Visualize the source datasets

The two panels show the CAGIS and OSM source lines in a common metric CRS using Matplotlib.

In [ ]:
metric_crs = cagis.estimate_utm_crs()
cagis_map = cagis.to_crs(metric_crs)
osm_map = osm.to_crs(metric_crs)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
cagis_map.plot(ax=axes[0], color="#2563eb", linewidth=0.8)
axes[0].set_title(f"CAGIS trails ({len(cagis_map):,} features)")
osm_map.plot(ax=axes[1], color="#ea580c", linewidth=0.8)
axes[1].set_title(f"OpenStreetMap trails ({len(osm_map):,} ways)")
for ax in axes:
    ax.set_axis_off()
    ax.set_aspect("equal", adjustable="datalim")
plt.show()

## 4. Prepare lines and visualize the tolerance buffers

The two panels show each source's centerlines over its configured tolerance buffer using Matplotlib.

In [ ]:
metric_crs = cagis.estimate_utm_crs()
cagis_lines = cagis.copy()
cagis_lines["geometry"] = shapely.force_2d(cagis_lines.geometry.values)
cagis_lines = cagis_lines.explode(index_parts=False).reset_index(drop=True)
cagis_lines = cagis_lines[cagis_lines.geometry.geom_type == "LineString"].to_crs(metric_crs)
cagis_lines = cagis_lines[~cagis_lines.geometry.is_empty & (cagis_lines.geometry.length > 0)].reset_index(drop=True)

osm_lines = osm.copy()
osm_lines["geometry"] = shapely.force_2d(osm_lines.geometry.values)
osm_lines = osm_lines.explode(index_parts=False).reset_index(drop=True)
osm_lines = osm_lines[osm_lines.geometry.geom_type == "LineString"].to_crs(metric_crs)
osm_lines = osm_lines[~osm_lines.geometry.is_empty & (osm_lines.geometry.length > 0)].reset_index(drop=True)

osm_match_candidates = osm.copy()
osm_match_candidates["geometry"] = shapely.force_2d(osm_match_candidates.geometry.values)
osm_match_candidates = osm_match_candidates.explode(index_parts=False).reset_index(drop=True)
osm_match_candidates = osm_match_candidates[
    osm_match_candidates.geometry.geom_type.isin({"LineString", "Polygon", "MultiPolygon"})
].to_crs(metric_crs)
osm_match_candidates = osm_match_candidates[~osm_match_candidates.geometry.is_empty].reset_index(drop=True)

cagis_buffer = cagis_lines[["geometry"]].copy()
cagis_buffer["geometry"] = cagis_lines.geometry.buffer(TOL_M)
osm_buffer = osm_lines[["geometry"]].copy()
osm_buffer["geometry"] = osm_lines.geometry.buffer(TOL_M)

print(f"Prepared line features: CAGIS={len(cagis_lines):,}, OSM={len(osm_lines):,}")
print(f"OSM match candidates including pedestrian areas: {len(osm_match_candidates):,}")
print(f"Metric CRS: {metric_crs}; tolerance: {TOL_M} m")

fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
cagis_buffer.plot(ax=axes[0], color="#60a5fa", alpha=0.35, edgecolor="#2563eb", linewidth=0.25)
cagis_lines.plot(ax=axes[0], color="#1e3a8a", linewidth=0.65)
axes[0].set_title(f"CAGIS lines + {TOL_M} m buffers")
osm_buffer.plot(ax=axes[1], color="#fdba74", alpha=0.4, edgecolor="#ea580c", linewidth=0.25)
osm_lines.plot(ax=axes[1], color="#9a3412", linewidth=0.65)
axes[1].set_title(f"OSM lines + {TOL_M} m buffers")
for ax in axes:
    ax.set_axis_off()
    ax.set_aspect("equal", adjustable="datalim")
plt.show()

## 5. Match runs and export neutral source layers

Each source line is divided into approximately `STEP_M` metre pieces. A piece matches the best individual opposite-source candidate covering at least `MIN_COVER` of its length. OSM line candidates use their `TOL_M` metre buffer; pedestrian areas use their polygon footprint directly, with no additional buffer. Matching is independent in each direction and need not be reciprocal. Adjacent pieces with the same counterpart are merged. All runs, including short runs and matches with identical tags, are preserved.

OSM pedestrian area ways and multipolygon relations are downloaded into the existing dated OSM cache. Sidewalks, crossings, and service roads are retained only when they have trail_visibility. Pedestrian areas are matching surfaces for CAGIS runs, not OSM line runs or rendered line layers.

Each run keeps its source geometry and attributes, plus the selected counterpart feature and part identifiers where matched. Step 5 does not copy counterpart tags. The neutral layers are exported to `output/cagis_runs.geojson` and `output/osm_runs.geojson` in EPSG:4326. Step 6 uses the identifiers to look up only the counterpart attributes needed for interpretation and writes separate interpreted exports.

## 6. Interpret matched tags and export interpreted run layers

This step resolves `counterpart_feature_id` and `counterpart_part` against the prepared source feature tables, compares only the tags needed by the documented rules, and writes `output/cagis_interpreted_runs.geojson` and `output/osm_interpreted_runs.geojson`. Step 5's neutral `*_runs.geojson` files retain source tags and match identifiers only; they are not enriched with the other source's tags. The interpreted exports include the derived flags and selected comparison fields for ArcGIS Online, QGIS, and other GIS software.

The evaluator does not repeat spatial matching. See [symbology_rules.md](symbology_rules.md) for portable predicates and styling. These categories are review candidates, not verified errors.

In [ ]:
def prepare_match_source(lines, dataset, identity_field):
    source = lines.reset_index(drop=True).copy()
    if dataset == "cagis":
        source = source.rename(columns={column: f"cagis_{column}" for column in source.columns if column != "geometry"})
        identity_field = f"cagis_{identity_field}"
        aliases = {"cagis_name": f"cagis_{name_field}" if name_field else None, "cagis_status": "cagis_STATUS"}
    else:
        aliases = {f"osm_{lifecycle}_highway": f"osm_{lifecycle}:highway" for lifecycle in ("proposed", "abandoned", "disused")}
    for alias, original in aliases.items():
        if alias in source.columns:
            raise ValueError(f"Source attribute collides with reserved alias: {alias}")
        source[alias] = source[original] if original in source.columns else None
    if source[identity_field].isna().any():
        raise ValueError(f"Missing feature identity in {identity_field}")
    identities = source[identity_field].map(str)
    parts = identities.groupby(identities, sort=False).cumcount()
    return source, identities.tolist(), parts.tolist()


def match_runs(source, other, dataset, source_ids, source_ordinals, other_ids, other_ordinals, tolerance_m, step_m, min_cover):
    if source.crs != other.crs or source.crs is None or source.crs.is_geographic:
        raise ValueError("Matching requires a shared projected metric CRS.")
    if tolerance_m <= 0 or step_m <= 0 or not 0 < min_cover <= 1:
        raise ValueError("Tolerance/step must be positive; coverage must be in (0, 1].")
    source_fields = [column for column in source.columns if column != "geometry"]
    if set(source_fields) & {"run_id", "counterpart_feature_id", "counterpart_part", "match_state", "match_metrics"}:
        raise ValueError("Source attributes collide with reserved matching fields.")
    metadata_fields = ["run_id", "counterpart_feature_id", "counterpart_part", "match_state", "match_metrics"]
    columns = metadata_fields + sorted(source_fields) + ["geometry"]
    other_geometries = other.geometry.values
    other_buffers = [
        geometry if geometry.geom_type in {"Polygon", "MultiPolygon"} else geometry.buffer(tolerance_m)
        for geometry in other_geometries
    ]
    tree = shapely.STRtree(other_buffers)
    source_attributes = source.drop(columns="geometry").to_dict("records")
    rows = []
    for source_index, line in enumerate(source.geometry):
        piece_count = max(1, int(round(line.length / step_m)))
        marks = [line.length * piece_index / piece_count for piece_index in range(piece_count + 1)]
        evaluated = []
        for piece_index in range(piece_count):
            piece = substring(line, marks[piece_index], marks[piece_index + 1])
            piece_buffer = piece.buffer(tolerance_m)
            scores = []
            for candidate in tree.query(piece, predicate="intersects"):
                candidate = int(candidate)
                coverage = shapely.intersection(piece, other_buffers[candidate]).length / piece.length
                if coverage >= min_cover:
                    overlap = shapely.intersection(piece_buffer, other_buffers[candidate]).area / piece_buffer.area
                    scores.append((
                        -overlap, -coverage, shapely.distance(piece, other_geometries[candidate]),
                        other_ids[candidate], other_ordinals[candidate], candidate, coverage,
                    ))
            best = min(scores) if scores else None
            evaluated.append((best[5] if best else None, best[6] if best else None, piece.length))
        start = 0
        run_number = 0
        while start < piece_count:
            counterpart = evaluated[start][0]
            end = start + 1
            while end < piece_count and evaluated[end][0] == counterpart:
                end += 1
            geometry = substring(line, marks[start], marks[end])
            coverage = (
                sum(evaluated[piece_index][1] * evaluated[piece_index][2] for piece_index in range(start, end))
                / sum(evaluated[piece_index][2] for piece_index in range(start, end))
                if counterpart is not None else None
            )
            match_metrics = {
                "start_m": marks[start],
                "end_m": marks[end],
                "run_length_m": geometry.length,
                "feature_length_m": line.length,
            }
            if coverage is not None:
                match_metrics["match_coverage"] = coverage
            rows.append({
                **source_attributes[source_index],
                "run_id": f"{dataset}:{source_ids[source_index]}:{source_ordinals[source_index]}:{run_number}",
                "counterpart_feature_id": other_ids[counterpart] if counterpart is not None else None,
                "counterpart_part": other_ordinals[counterpart] if counterpart is not None else None,
                "match_state": "matched" if counterpart is not None else "unmatched",
                "match_metrics": match_metrics,
                "geometry": geometry,
            })
            start = end
            run_number += 1
    return gpd.GeoDataFrame(rows, columns=columns, geometry="geometry", crs=source.crs)


def write_run_layer(runs, path):
    geographic = runs.to_crs("EPSG:4326")
    payload = geojson_payload_without_nulls(geographic)
    payload["run_columns"] = [column for column in runs.columns if column != "geometry"]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, allow_nan=False), encoding="utf-8")


def read_run_layer(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    columns = payload["run_columns"] + ["geometry"]
    if payload["features"]:
        layer = gpd.GeoDataFrame.from_features(payload["features"], crs="EPSG:4326")
        return layer.reindex(columns=columns)
    return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs="EPSG:4326")


cagis_source, cagis_ids, cagis_parts = prepare_match_source(cagis_lines, "cagis", id_field)
osm_candidate_source, osm_candidate_ids, osm_candidate_parts = prepare_match_source(
    osm_match_candidates, "osm", "osm_id",
)
osm_line_mask = osm_candidate_source.geometry.geom_type.eq("LineString").tolist()
osm_source = osm_candidate_source.loc[osm_line_mask].reset_index(drop=True)
osm_ids = [identity for identity, is_line in zip(osm_candidate_ids, osm_line_mask) if is_line]
osm_parts = [part for part, is_line in zip(osm_candidate_parts, osm_line_mask) if is_line]
cagis_runs = match_runs(
    cagis_source, osm_candidate_source, "cagis", cagis_ids, cagis_parts,
    osm_candidate_ids, osm_candidate_parts, TOL_M, STEP_M, MIN_COVER,
)
osm_runs = match_runs(
    osm_source, cagis_source, "osm", osm_ids, osm_parts, cagis_ids, cagis_parts, TOL_M, STEP_M, MIN_COVER,
)

OUTPUT_DIR = PROJECT_DIR / "output"
for dataset, runs in (("cagis", cagis_runs), ("osm", osm_runs)):
    path = OUTPUT_DIR / f"{dataset}_runs.geojson"
    write_run_layer(runs, path)
    print(f"{dataset.upper()}: {len(runs):,} runs exported to {path}")
    for state in ("matched", "unmatched"):
        selected = runs.loc[runs["match_state"] == state]
        total_length = selected['match_metrics'].map(lambda metrics: metrics['run_length_m']).sum()
        print(f"  {state}: {len(selected):,} runs, {total_length / 1000:.2f} km")

In [ ]:
import re
import pandas as pd


def tag_text(value):
    return "" if pd.isna(value) else str(value).strip()


def cagis_effective_status(status, name=None):
    return "Proposed" if "proposed" in tag_text(name).casefold() else tag_text(status)


def feature_attribute_lookup(source, feature_ids, feature_parts):
    attributes = source.drop(columns="geometry").to_dict("records")
    return {
        (str(feature_id), int(feature_part)): row
        for feature_id, feature_part, row in zip(feature_ids, feature_parts, attributes)
    }


def run_source_info(run_id):
    dataset_label, feature_key, part_index, _ = run_id.split(":", 3)
    return dataset_label, feature_key, int(part_index)


def counterpart_attributes(row, cagis_lookup, osm_lookup):
    if row["match_state"] != "matched":
        return None
    dataset_label, _, _ = run_source_info(row["run_id"])
    lookup = osm_lookup if dataset_label == "cagis" else cagis_lookup
    key = (str(row["counterpart_feature_id"]), int(row["counterpart_part"]))
    attributes = lookup.get(key)
    if attributes is None:
        raise ValueError(f"Missing counterpart feature for {key}")
    return attributes


def osm_lifecycle(row):
    highway = tag_text(row.get("osm_highway")).casefold()
    proposed = bool(tag_text(row.get("osm_proposed_highway"))) or highway == "proposed"
    retired = bool(tag_text(row.get("osm_abandoned_highway")) or tag_text(row.get("osm_disused_highway"))) or highway in {"abandoned", "disused"}
    return proposed, retired


def status_discrepancy(cagis_status, osm_row, cagis_name=None):
    status = tag_text(cagis_status)
    key = cagis_effective_status(status, cagis_name).casefold()
    if key not in {"open", "proposed", "closed", "non-operational"}:
        return None
    proposed, retired = osm_lifecycle(osm_row)
    if key == "open":
        compatible = not proposed and not retired
    elif key == "proposed":
        compatible = proposed and not retired
    elif key == "closed":
        effective_access = tag_text(osm_row.get("osm_foot")).casefold() or tag_text(osm_row.get("osm_access")).casefold()
        compatible = (retired or effective_access in {"no", "private"}) and not proposed
    else:
        compatible = retired and not proposed
    if compatible:
        return None
    tags = []
    for field, label in (
        ("osm_highway", "highway"), ("osm_proposed_highway", "proposed:highway"),
        ("osm_abandoned_highway", "abandoned:highway"), ("osm_disused_highway", "disused:highway"),
    ):
        value = tag_text(osm_row.get(field))
        if value:
            tags.append(f"{label}={value}")
    osm_status = ", ".join(tags)
    return f"status_mismatch (CAGIS STATUS={status or '?'}, OSM {osm_status or '?'})"


def access_discrepancy(cagis_status, osm_row, cagis_name=None):
    status = tag_text(cagis_status)
    key = cagis_effective_status(status, cagis_name).casefold()
    if key not in {"open", "proposed", "closed", "non-operational"}:
        return None
    access = tag_text(osm_row.get("osm_access")).casefold()
    foot = tag_text(osm_row.get("osm_foot")).casefold()
    effective_access = foot or access
    if effective_access == "unknown":
        compatible = False
    elif key == "open":
        compatible = effective_access in {"", "yes", "designated"}
    elif key == "proposed":
        compatible = True
    else:
        compatible = effective_access not in {"yes", "designated"}
    if compatible:
        return None
    return f"access_mismatch (CAGIS STATUS={status or '?'}, OSM access={access or '?'}, foot={foot or '?'})"


def name_discrepancy(cagis_value, osm_value):
    names = [tag_text(value) for value in (cagis_value, osm_value)]
    names = ["" if name.casefold() in {"none", "nan", "null", "tbd", "unknown"} else name for name in names]
    cagis_name, osm_name = names
    if bool(cagis_name) != bool(osm_name):
        return "missing_name"
    if not cagis_name or cagis_name.startswith(osm_name) or osm_name.startswith(cagis_name):
        return None
    normalized = []
    for name in names:
        name = re.sub(r"([a-z])([A-Z])", r"\1 \2", name).lower()
        name = re.sub(r"[\s\-_/]+", " ", name)
        name = re.sub(r"\b(trail|trai|path|loop|connector|route)\b", "", name)
        normalized.append(re.sub(r"\s+", " ", name).strip())
    return "name_mismatch" if normalized[0] != normalized[1] else None


def interpret_runs(runs, cagis_lookup, osm_lookup):
    required = {"run_id", "match_state", "counterpart_feature_id", "counterpart_part"}
    if not required.issubset(runs.columns):
        raise ValueError(f"Missing run fields: {sorted(required - set(runs.columns))}")
    if not runs["match_state"].isin({"matched", "unmatched"}).all():
        raise ValueError("Unknown source dataset or match state.")
    interpreted = runs.copy()
    flags = []
    for _, row in runs.iterrows():
        dataset_label, _, _ = run_source_info(row["run_id"])
        source_proposed = cagis_effective_status(row.get("cagis_status"), row.get("cagis_name")).casefold() == "proposed" if dataset_label == "cagis" else osm_lifecycle(row)[0]
        name_issue = None
        status_issue = None
        access_issue = None
        reasons = []
        if row["match_state"] == "matched":
            counterpart = counterpart_attributes(row, cagis_lookup, osm_lookup)
            if dataset_label == "cagis":
                name_issue = name_discrepancy(row.get("cagis_name"), counterpart.get("osm_name"))
                status_issue = status_discrepancy(row.get("cagis_status"), counterpart, row.get("cagis_name"))
                access_issue = access_discrepancy(row.get("cagis_status"), counterpart, row.get("cagis_name"))
                if name_issue:
                    reasons.append(f"{name_issue} (CAGIS='{tag_text(row.get('cagis_name'))}', OSM='{tag_text(counterpart.get('osm_name'))}')")
            else:
                name_issue = name_discrepancy(counterpart.get("cagis_name"), row.get("osm_name"))
                status_issue = status_discrepancy(counterpart.get("cagis_status"), row, counterpart.get("cagis_name"))
                access_issue = access_discrepancy(counterpart.get("cagis_status"), row, counterpart.get("cagis_name"))
                if name_issue:
                    reasons.append(f"{name_issue} (CAGIS='{tag_text(counterpart.get('cagis_name'))}', OSM='{tag_text(row.get('osm_name'))}')")
            if status_issue:
                reasons.append(status_issue)
            if access_issue:
                reasons.append(access_issue)
            category = "tag_discrepancy" if reasons else "matched"
        elif dataset_label == "cagis":
            category = "cagis_only"
        else:
            active = status_discrepancy("Open", row) is None
            effective_access = tag_text(row.get("osm_foot")).casefold() or tag_text(row.get("osm_access")).casefold()
            visibility = tag_text(row.get("osm_trail_visibility")).casefold()
            eligible = active and effective_access in {"", "yes", "designated", "permissive", "unknown"} and visibility != "no"
            category = "osm_only" if eligible else "hidden"
        flags.append({
            "name_mismatch": name_issue is not None,
            "status_mismatch": status_issue is not None,
            "access_mismatch": access_issue is not None,
            "discrepancy_reasons": "; ".join(sorted(reasons)),
            "source_is_proposed": bool(source_proposed),
            "display_category": category,
        })
    for field, dtype in (
        ("name_mismatch", bool), ("status_mismatch", bool), ("access_mismatch", bool), ("discrepancy_reasons", str),
        ("source_is_proposed", bool), ("display_category", str),
    ):
        interpreted[field] = pd.Series([flag[field] for flag in flags], index=runs.index, dtype=dtype)
    return interpreted


cagis_lookup = feature_attribute_lookup(cagis_source, cagis_ids, cagis_parts)
osm_lookup = feature_attribute_lookup(osm_candidate_source, osm_candidate_ids, osm_candidate_parts)
cagis_display = interpret_runs(cagis_runs, cagis_lookup, osm_lookup)
osm_display = interpret_runs(osm_runs, cagis_lookup, osm_lookup)
for label, layer in (("CAGIS", cagis_display), ("OSM", osm_display)):
    print(f"{label} interpretation: {layer['display_category'].value_counts().to_dict()}")

In [ ]:
for label, layer in (("CAGIS", cagis_display), ("OSM", osm_display)):
    path = OUTPUT_DIR / f"{label.lower()}_interpreted_runs.geojson"
    export = layer.copy()
    if "match_metrics" in export.columns:
        metrics = export["match_metrics"]
        export["match_start_m"] = metrics.map(lambda value: value.get("start_m") if isinstance(value, dict) else None)
        export["match_end_m"] = metrics.map(lambda value: value.get("end_m") if isinstance(value, dict) else None)
        export["match_run_length_m"] = metrics.map(lambda value: value.get("run_length_m") if isinstance(value, dict) else None)
        export["match_feature_length_m"] = metrics.map(lambda value: value.get("feature_length_m") if isinstance(value, dict) else None)
        export["match_coverage"] = metrics.map(lambda value: value.get("match_coverage") if isinstance(value, dict) else None)
        export["match_metrics_json"] = metrics.map(lambda value: json.dumps(value, sort_keys=True) if isinstance(value, dict) else None)
        export = export.drop(columns=["match_metrics"])
    write_run_layer(export, path)
    print(f"{label} interpreted runs exported to {path}")


## 7. Visualize interpreted runs with Plotly

The Plotly rendering adapter below reads step 6's interpreted GeoJSON files. This uses the same stored fields available to other GIS tools; only temporary display geometries are modified for dashed styling.

In [ ]:
cagis_display = read_run_layer(OUTPUT_DIR / "cagis_interpreted_runs.geojson")
osm_display = read_run_layer(OUTPUT_DIR / "osm_interpreted_runs.geojson")

In [ ]:
import html
import textwrap

from shapely.geometry import MultiLineString
from shapely.ops import substring


def iter_plot_paths(geometry):
    if geometry is None or geometry.is_empty:
        return
    if geometry.geom_type == "LineString":
        yield list(geometry.coords)
    elif geometry.geom_type == "Polygon":
        yield list(geometry.exterior.coords)
    elif hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            yield from iter_plot_paths(part)


def feature_hover(row):
    parts = []
    for key, value in row.items():
        if key == "geometry" or pd.isna(value) or str(value) == "":
            continue
        wrapped = []
        for text in (key, value):
            lines = textwrap.wrap(str(text), width=42, break_long_words=True, break_on_hyphens=False)
            wrapped.append("<br>".join(html.escape(line) for line in lines))
        parts.append(f"<b>{wrapped[0]}</b>: {wrapped[1]}")
    return "<br>".join(parts) if parts else html.escape(str(row.name))


def add_satellite_layer(fig, gdf, name, color, width=2, opacity=1.0):
    latitudes, longitudes, hovertext = [], [], []
    proposed = gdf["source_is_proposed"].fillna(False).tolist()
    if any(proposed):
        display = gdf.to_crs(gdf.estimate_utm_crs()).copy()
        geometries = []
        for geometry, is_proposed in zip(display.geometry, proposed):
            if not is_proposed or geometry is None or geometry.is_empty:
                geometries.append(geometry)
                continue
            lines = [geometry] if geometry.geom_type == "LineString" else geometry.geoms
            dashes = []
            for line in lines:
                start = 0
                while start < line.length:
                    dashes.append(substring(line, start, min(start + 5, line.length)))
                    start += 15
            geometries.append(MultiLineString(dashes))
        display["geometry"] = geometries
        geographic = display.to_crs("EPSG:4326")
    else:
        geographic = gdf.to_crs("EPSG:4326")
    for _, row in geographic.iterrows():
        hover = feature_hover(row)
        for path in iter_plot_paths(row.geometry):
            longitudes.extend(point[0] for point in path)
            latitudes.extend(point[1] for point in path)
            hovertext.extend([hover] * len(path))
            longitudes.append(None)
            latitudes.append(None)
            hovertext.append(None)
    fig.add_trace(go.Scattermap(
        lat=latitudes, lon=longitudes, text=hovertext, mode="lines", name=name,
        line={"color": color, "width": width}, opacity=opacity,
        hovertemplate="%{text}<extra></extra>",
        hoverlabel={"align": "left", "font": {"size": 11}}, connectgaps=False,
    ))


def add_polygon_layer(fig, gdf, name, line_color, fillcolor=None, width=1, simplify_m=1, hover_properties=False):
    geoms = gdf.geometry.simplify(simplify_m) if simplify_m else gdf.geometry
    geographic = gdf.set_geometry(geoms).to_crs("EPSG:4326")
    if hover_properties and not geographic.empty:
        fig.add_trace(go.Choroplethmap(
            geojson=geographic.__geo_interface__,
            locations=[str(index) for index in geographic.index], z=[0] * len(geographic),
            text=[feature_hover(row) for _, row in geographic.iterrows()],
            colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
            marker_line_width=0, showscale=False, showlegend=False,
            name=name, legendgroup=name, hovertemplate="%{text}<extra></extra>",
            hoverlabel={"align": "left", "font": {"size": 11}}, visible="legendonly",
        ))
    latitudes, longitudes, hovertext = [], [], []
    for _, row in geographic.iterrows():
        hover = feature_hover(row) if hover_properties else None
        for path in iter_plot_paths(row.geometry):
            longitudes.extend(point[0] for point in path)
            latitudes.extend(point[1] for point in path)
            hovertext.extend([hover] * len(path))
            longitudes.append(None)
            latitudes.append(None)
            hovertext.append(None)
    fig.add_trace(go.Scattermap(
        lat=latitudes, lon=longitudes, text=hovertext if hover_properties else None,
        mode="lines", name=name, legendgroup=name,
        line={"color": line_color, "width": width},
        fill="toself" if fillcolor else "none", fillcolor=fillcolor,
        hoverinfo="text" if hover_properties else "skip",
        hovertemplate="%{text}<extra></extra>" if hover_properties else None,
        hoverlabel={"align": "left", "font": {"size": 11}}, visible="legendonly",
    ))


fig = go.Figure()
fig.update_layout(
    title="Candidate trail discrepancies - satellite imagery", height=750,
    margin={"l": 5, "r": 5, "t": 55, "b": 5},
    legend={"orientation": "h", "yanchor": "bottom", "y": -0.2, "xanchor": "left", "x": 0},
    map={
        "style": "white-bg", "center": {"lat": 39.14, "lon": -84.49}, "zoom": 10,
        "layers": [{
            "below": "traces", "sourcetype": "raster",
            "source": ["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"],
            "sourceattribution": "Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        }],
    },
)

comparison_parks = select_comparison_parks(parks, cagis)
if not comparison_parks.empty:
    comparison_parks_buffer = comparison_parks.to_crs(comparison_parks.estimate_utm_crs())
    comparison_parks_buffer = comparison_parks_buffer.set_geometry(comparison_parks_buffer.geometry.buffer(PARKS_BUFFER_M))
    add_polygon_layer(fig, comparison_parks_buffer, "CPB-maintained parks or parks with CAGIS trails", "#4ade80", width=2, simplify_m=0, hover_properties=True)
for label, layer, edge, fill in (
    ("CAGIS", cagis_display, "rgba(147,197,253,0.6)", "rgba(147,197,253,0.3)"),
    ("OSM", osm_display, "rgba(253,186,116,0.6)", "rgba(253,186,116,0.3)"),
):
    if layer.empty:
        continue
    buffer_layer = layer.to_crs(layer.estimate_utm_crs())
    buffer_layer = buffer_layer.set_geometry(buffer_layer.geometry.buffer(TOL_M))
    add_polygon_layer(fig, buffer_layer[["geometry"]], f"{label} matching buffer", edge, fill)

add_satellite_layer(fig, cagis_display, "CAGIS context", "#feefdb", width=1, opacity=0.5)
add_satellite_layer(fig, osm_display, "OSM context", "#feefdb", width=1, opacity=0.5)

for layer, category, label, color, width in (
    (cagis_display, "cagis_only", "Only on CAGIS", "#30ffd2", 1),
    (osm_display, "osm_only", "Only on OSM", "#ffd429", 1),
):
    selected = layer.loc[layer["display_category"].eq(category)]
    add_satellite_layer(fig, selected, label, color, width=width)

for flag, label, color in (
    ("name_mismatch", "Name discrepancy", "#f93232"),
    ("access_mismatch", "Access discrepancy", "#ff981a"),
    ("status_mismatch", "Status discrepancy", "#f93232"),
):
    selected = cagis_display.loc[cagis_display[flag]]
    if not selected.empty:
        add_satellite_layer(fig, selected, label, color, width=2)
fig.update_traces(selector={"name": "Name discrepancy"}, visible="legendonly")
fig.show()